# CS1 SCoPE2 Preprocessing -- Regenerate `abstracted_code_v1`

## 1. Dependencies

In [ ]:
import sys
import subprocess

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "pandas",
    "pyarrow",
    "numpy",
    "matplotlib",
    "tqdm",
    "psutil",
    "tree-sitter==0.23.0",
    "tree-sitter-cpp",
    "ruamel.yaml>=0.2.7",
], check=True)

print("Base dependencies installed.")


## 2. Workspace and data paths

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
REPO_BRANCH = "simo"

SCOPE2_REPO_URL = "https://github.com/jp2425/SCoPE2.git"
SCOPE2_BRANCH = "main"

WORKSPACE_ROOT = Path.cwd()

REPO_ROOT = WORKSPACE_ROOT / "DiverseVul--IS-Project"
PROJECT_DIR = REPO_ROOT / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

SCOPE2_SRC_DIR = WORKSPACE_ROOT / "SCoPE2_source"

DATA_ROOT = WORKSPACE_ROOT / "IntelligentSystemProject" / "VulnerabilityDetectionData"
PROCESSED_DIR = DATA_ROOT / "processed"
OUTPUT_ROOT = DATA_ROOT / "outputs"

INPUT_PARQUET = PROCESSED_DIR / "rdiversevul_cs1_normalized_plus_abstracted_v1.parquet"
OUTPUT_PARQUET = PROCESSED_DIR / "rdiversevul_cs1_normalized_plus_abstracted_v2.parquet"

PREPROCESSING_OUTPUT_DIR = OUTPUT_ROOT / "case_study_1" / "scope2_preprocessing_v2"
PREPROCESSING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_SIZE = 30
RANDOM_STATE = 42
LOG_EVERY = 25_000

RUN_FULL_DATASET = True

STORAGE_CAP_GB = 60

print("WORKSPACE_ROOT:", WORKSPACE_ROOT)
print("INPUT_PARQUET:", INPUT_PARQUET)
print("OUTPUT_PARQUET:", OUTPUT_PARQUET)


## 3. Fetch repositories

In [ ]:
import urllib.request
import zipfile


def download_and_extract_repo(repo_url: str, branch: str, target_dir: Path) -> None:
    if target_dir.exists():
        print(f"Repository already exists at {target_dir}")
        return

    print(f"Downloading {repo_url} (branch: {branch}) without git...")
    clean_url = repo_url.removesuffix(".git")
    zip_url = f"{clean_url}/archive/refs/heads/{branch}.zip"
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    zip_path = target_dir.parent / f"{target_dir.name}_download_temp.zip"

    urllib.request.urlretrieve(zip_url, zip_path)

    print("Extracting files...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(target_dir.parent)

    repo_name = clean_url.split("/")[-1]
    extracted_folder = target_dir.parent / f"{repo_name}-{branch}"
    if extracted_folder.exists():
        extracted_folder.rename(target_dir)

    zip_path.unlink()
    print(f"Repository ready at {target_dir}")


download_and_extract_repo(REPO_URL, REPO_BRANCH, REPO_ROOT)
download_and_extract_repo(SCOPE2_REPO_URL, SCOPE2_BRANCH, SCOPE2_SRC_DIR)


## 4. Install SCoPE2

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(SCOPE2_SRC_DIR)], check=True)

import tree_sitter_cpp
from tree_sitter import Language
from SCoPE2.SCoPE import SCoPE
from SCoPE2.representations.CodeRepresentation import CodeRepresentation

print("SCoPE2 import OK.")


## 5. Load dataset

In [ ]:
import pandas as pd

required_data_files = {"input parquet (v1)": INPUT_PARQUET}
missing = {name: path for name, path in required_data_files.items() if not path.is_file()}
if missing:
    print("Missing required data files:")
    for name, path in missing.items():
        print(f"  - {name}: {path}")
    raise FileNotFoundError(
        "The v1 parquet is missing. It is produced by the Case Study 1 pipeline "
        "(normalization_v3.py) -- copy it into place before running this notebook."
    )

full_df = pd.read_parquet(INPUT_PARQUET)
print("full_df rows:", len(full_df))
print("full_df columns:", list(full_df.columns))

required_columns = ["source_row_id", "label", "project", "normalized_code", "abstracted_code_v1"]
missing_columns = [c for c in required_columns if c not in full_df.columns]
if missing_columns:
    raise KeyError(f"Missing expected columns in {INPUT_PARQUET}: {missing_columns}")

print(full_df[["source_row_id", "label", "project"]].head())


## 6. Write scope2_preprocessing.py

In [ ]:
src_cs1_dir = SRC_DIR / "case_study_1"
src_cs1_dir.mkdir(parents=True, exist_ok=True)
(src_cs1_dir / "scope2_preprocessing.py").write_text(
    'from __future__ import annotations\n\nfrom collections import Counter\nfrom dataclasses import dataclass\nimport logging\nimport time\nimport warnings\nfrom typing import Iterable, List, NamedTuple, Optional, Sequence, Type, Union\n\nimport pandas as pd\n\nlogger = logging.getLogger(__name__)\n\nSCOPE2_ABSTRACTION_VERSION = "cs1-scope2-abstraction-v2"\n\nDEFAULT_SOURCE_COLUMN = "normalized_code"\nDEFAULT_TARGET_COLUMN = "abstracted_code_v1"\nDEFAULT_FAILURE_FLAG_COLUMN = "abstracted_code_v1_scope2_failed"\n\nMAX_INPUT_SIZE = 10_000_000\n\n_INSTALL_HINT = (\n    "SCoPE2 is not installed (it is not on PyPI, must be installed from source). Run:\\n"\n    "  pip install \'tree-sitter==0.23.0\' tree-sitter-cpp \'ruamel.yaml>=0.2.7\'\\n"\n    "  pip install git+https://github.com/jp2425/SCoPE2.git\\n"\n    "tree-sitter-cpp is a separate grammar package SCoPE2 does not declare as a "\n    "dependency in its own pyproject.toml; it must be installed explicitly."\n)\n\n_SCOPE2_NOT_FOUND_MARKER = "TreeSitter didn\'t returned any "\n\n\ndef _import_scope2():\n    try:\n        import tree_sitter_cpp\n        from tree_sitter import Language\n        from SCoPE2.SCoPE import SCoPE\n        from SCoPE2.representations.CodeRepresentation import CodeRepresentation\n        from SCoPE2.transformation.implementations import (\n            RemoveCommentsTr,\n            ReplaceFunctionNamesTr,\n            ReplaceVariableNamesTr,\n        )\n    except ImportError as exc:\n        raise ImportError(_INSTALL_HINT) from exc\n\n    transformations = (RemoveCommentsTr, ReplaceVariableNamesTr, ReplaceFunctionNamesTr)\n    return SCoPE, Language, tree_sitter_cpp, CodeRepresentation, transformations\n\n\ndef build_scope2_processor(query_yaml_text: str):\n    SCoPE, Language, tree_sitter_cpp, _, _ = _import_scope2()\n    return SCoPE(query_yaml_text, Language(tree_sitter_cpp.language()))\n\n\ndef default_transformations() -> List[Type]:\n    _, _, _, _, transformations = _import_scope2()\n    return list(transformations)\n\n\ndef default_representation_cls():\n    _, _, _, code_representation_cls, _ = _import_scope2()\n    return code_representation_cls\n\n\ndef _coerce_code(value: object, max_size: int = MAX_INPUT_SIZE) -> str:\n    if value is None:\n        return ""\n    try:\n        if pd.isna(value):\n            return ""\n    except (TypeError, ValueError):\n        pass\n    if isinstance(value, bytes):\n        text = value.decode("utf-8", errors="replace")\n    elif isinstance(value, str):\n        text = value\n    else:\n        raise TypeError(f"Expected str or bytes source code, got {type(value).__name__}")\n    if len(text) > max_size:\n        raise ValueError(f"Code sample exceeds maximum size {max_size}: {len(text)}")\n    return text\n\n\ndef _extract_skip_query_id(warning_message: str) -> str:\n    idx = warning_message.find(_SCOPE2_NOT_FOUND_MARKER)\n    if idx == -1:\n        return warning_message\n    return warning_message[idx + len(_SCOPE2_NOT_FOUND_MARKER):].strip()\n\n\nclass Scope2RowResult(NamedTuple):\n    text: str\n    failed: bool\n    failure_reason: Optional[str]\n    input_was_empty: bool\n    skipped_transformations: tuple[str, ...]\n\n\ndef apply_scope2_to_code(\n    code: object,\n    scope,\n    transformations: Sequence[Type],\n    representation_cls: Type,\n) -> Scope2RowResult:\n    try:\n        text = _coerce_code(code)\n    except (TypeError, ValueError) as exc:\n        reason = f"{type(exc).__name__}: {exc}"\n        logger.warning("SCoPE2 input coercion failed; row flagged as failed. Reason: %s", reason)\n        return Scope2RowResult(text="", failed=True, failure_reason=reason, input_was_empty=False, skipped_transformations=())\n\n    if not text.strip():\n        return Scope2RowResult(text="", failed=False, failure_reason=None, input_was_empty=True, skipped_transformations=())\n\n    with warnings.catch_warnings(record=True) as caught:\n        warnings.simplefilter("always")\n        try:\n            transformed = scope.process(text, list(transformations), representation_cls)\n        except Exception as exc:\n            reason = f"{type(exc).__name__}: {exc}"\n            logger.warning("SCoPE2 failed on a row; falling back to normalized_code. Reason: %s", reason)\n            return Scope2RowResult(text=text, failed=True, failure_reason=reason, input_was_empty=False, skipped_transformations=())\n\n    skipped = tuple(_extract_skip_query_id(str(w.message)) for w in caught)\n    return Scope2RowResult(\n        text=str(transformed), failed=False, failure_reason=None, input_was_empty=False, skipped_transformations=skipped\n    )\n\n\n@dataclass(frozen=True)\nclass Scope2BatchReport:\n    n_rows: int\n    n_empty_input: int\n    n_failed: int\n    failure_reason_counts: Counter\n    skip_reason_counts: Counter\n    elapsed_seconds: float\n    failed_row_positions: tuple[int, ...]\n\n    def summary_lines(self) -> List[str]:\n        lines = [\n            f"SCoPE2 batch: {self.n_rows} rows, {self.elapsed_seconds:.1f}s "\n            f"({self.n_rows / self.elapsed_seconds if self.elapsed_seconds > 0 else float(\'inf\'):.1f} rows/sec)",\n            f"  empty input rows (passed through as empty string): {self.n_empty_input}",\n            f"  rows where SCoPE2 itself failed (fell back to normalized_code): {self.n_failed}",\n        ]\n        if self.failure_reason_counts:\n            lines.append(f"  failure reasons: {dict(self.failure_reason_counts)}")\n        if self.skip_reason_counts:\n            lines.append(\n                "  benign per-transformation skips (e.g. \'comment\' = rows with no comments to remove, "\n                f"NOT row failures): {dict(self.skip_reason_counts)}"\n            )\n        return lines\n\n\ndef apply_scope2_to_series(\n    codes: Union[pd.Series, Iterable[object]],\n    scope,\n    transformations: Optional[Sequence[Type]] = None,\n    representation_cls: Optional[Type] = None,\n    *,\n    log_every: int = 25_000,\n) -> tuple[pd.Series, pd.Series, Scope2BatchReport]:\n    if transformations is None:\n        transformations = default_transformations()\n    if representation_cls is None:\n        representation_cls = default_representation_cls()\n\n    if not isinstance(codes, pd.Series):\n        codes = pd.Series(list(codes))\n\n    n_rows = len(codes)\n    texts: List[str] = []\n    failed_flags: List[bool] = []\n    failure_reason_counts: Counter = Counter()\n    skip_reason_counts: Counter = Counter()\n    failed_positions: List[int] = []\n    n_empty_input = 0\n\n    start = time.monotonic()\n    for pos, value in enumerate(codes):\n        result = apply_scope2_to_code(value, scope, transformations, representation_cls)\n        texts.append(result.text)\n        failed_flags.append(result.failed)\n        if result.input_was_empty:\n            n_empty_input += 1\n        if result.failed:\n            failed_positions.append(pos)\n            reason_key = result.failure_reason.split(":", 1)[0] if result.failure_reason else "Unknown"\n            failure_reason_counts[reason_key] += 1\n        for query_id in result.skipped_transformations:\n            skip_reason_counts[query_id] += 1\n\n        if log_every and (pos + 1) % log_every == 0:\n            elapsed = time.monotonic() - start\n            rate = (pos + 1) / elapsed if elapsed > 0 else float("inf")\n            logger.info(\n                "SCoPE2 progress: %d/%d rows (%.1f rows/sec, %d failed so far)",\n                pos + 1, n_rows, rate, len(failed_positions),\n            )\n\n    elapsed_seconds = time.monotonic() - start\n\n    report = Scope2BatchReport(\n        n_rows=n_rows,\n        n_empty_input=n_empty_input,\n        n_failed=len(failed_positions),\n        failure_reason_counts=failure_reason_counts,\n        skip_reason_counts=skip_reason_counts,\n        elapsed_seconds=elapsed_seconds,\n        failed_row_positions=tuple(failed_positions[:100]),\n    )\n\n    abstracted_series = pd.Series(texts, index=codes.index, name=DEFAULT_TARGET_COLUMN)\n    failed_series = pd.Series(failed_flags, index=codes.index, name=DEFAULT_FAILURE_FLAG_COLUMN)\n    return abstracted_series, failed_series, report\n\n\ndef add_scope2_abstracted_code_column(\n    frame: pd.DataFrame,\n    scope,\n    source_column: str = DEFAULT_SOURCE_COLUMN,\n    target_column: str = DEFAULT_TARGET_COLUMN,\n    failure_flag_column: str = DEFAULT_FAILURE_FLAG_COLUMN,\n    transformations: Optional[Sequence[Type]] = None,\n    representation_cls: Optional[Type] = None,\n    *,\n    log_every: int = 25_000,\n) -> tuple[pd.DataFrame, Scope2BatchReport]:\n    if source_column not in frame.columns:\n        raise KeyError(f"Missing source column: {source_column}")\n\n    abstracted, failed, report = apply_scope2_to_series(\n        frame[source_column],\n        scope,\n        transformations=transformations,\n        representation_cls=representation_cls,\n        log_every=log_every,\n    )\n\n    output = frame.copy()\n    output[target_column] = abstracted.to_numpy()\n    output[failure_flag_column] = failed.to_numpy()\n    return output, report\n',
    encoding="utf-8",
)
print("Wrote", src_cs1_dir / "scope2_preprocessing.py")


## 7. Build SCoPE2 processor

In [ ]:
import sys

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from case_study_1 import scope2_preprocessing as s2p

query_yaml_path = SCOPE2_SRC_DIR / "SCoPE2" / "query" / "cpp.yaml"
query_yaml_text = query_yaml_path.read_text(encoding="utf-8")

scope = s2p.build_scope2_processor(query_yaml_text)
transformations = s2p.default_transformations()
representation_cls = s2p.default_representation_cls()

print("SCoPE2 processor ready. Transformations:", [t.__name__ for t in transformations])


## 8. Smoke test sample

In [ ]:
sample_df = full_df.sample(n=min(SAMPLE_SIZE, len(full_df)), random_state=RANDOM_STATE)

sample_abstracted, sample_failed, sample_report = s2p.apply_scope2_to_series(
    sample_df["normalized_code"], scope, transformations, representation_cls, log_every=0,
)

print("\n".join(sample_report.summary_lines()))


## 9. Preview before/after

In [ ]:
preview_df = sample_df.copy()
preview_df["abstracted_code_v1_scope2"] = sample_abstracted.to_numpy()
preview_df["scope2_failed"] = sample_failed.to_numpy()

PREVIEW_N = 5
for _, row in preview_df.head(PREVIEW_N).iterrows():
    print("=" * 80)
    print("source_row_id:", row["source_row_id"], "| scope2_failed:", row["scope2_failed"])
    print("--- BEFORE (normalized_code) ---")
    print(row["normalized_code"][:500])
    print("--- AFTER (SCoPE2 abstracted_code_v1) ---")
    print(row["abstracted_code_v1_scope2"][:500])

print("\nReview the before/after pairs above before proceeding to the full-dataset pass.")


## 10. Full-dataset pass

In [ ]:
if RUN_FULL_DATASET:
    full_abstracted, full_failed, full_report = s2p.apply_scope2_to_series(
        full_df["normalized_code"], scope, transformations, representation_cls, log_every=LOG_EVERY,
    )
    print("\n".join(full_report.summary_lines()))
else:
    full_abstracted = full_failed = full_report = None
    print("RUN_FULL_DATASET=False; skipping the full-dataset pass.")


## 11. Save output parquet

In [ ]:
if full_abstracted is None:
    raise RuntimeError("Run the full-dataset pass above (RUN_FULL_DATASET=True) before saving output.")

output_df = full_df.copy()
output_df["abstracted_code_v1"] = full_abstracted.to_numpy()
output_df["abstracted_code_v1_scope2_failed"] = full_failed.to_numpy()

assert len(output_df) == len(full_df), "Row count changed during SCoPE2 processing."
assert output_df["abstracted_code_v1"].isna().sum() == 0, "Null values introduced in abstracted_code_v1."

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
output_df.to_parquet(OUTPUT_PARQUET, index=False)

size_mb = OUTPUT_PARQUET.stat().st_size / 1e6
print(f"Saved {OUTPUT_PARQUET} ({size_mb:.1f} MB, {len(output_df)} rows)")


## 12. Sanity checks

In [ ]:
from case_study_1.normalization_v3 import representation_summary

print("v1 abstracted_code_v1 summary (old, regex-based):")
print(representation_summary(full_df["abstracted_code_v1"]))
print()
print("v2 abstracted_code_v1 summary (new, SCoPE2-based):")
print(representation_summary(output_df["abstracted_code_v1"]))
print()
print("Row count unchanged:", len(output_df) == len(full_df))
print("Null count in v2 abstracted_code_v1:", output_df["abstracted_code_v1"].isna().sum())
print("SCoPE2 failed-row rate:", output_df["abstracted_code_v1_scope2_failed"].mean())


In [ ]:
random_check = output_df.sample(5, random_state=123)
for _, row in random_check.iterrows():
    print("=" * 80)
    print("source_row_id:", row["source_row_id"])
    print("--- normalized_code ---")
    print(row["normalized_code"][:400])
    print("--- abstracted_code_v1 (v2, SCoPE2) ---")
    print(row["abstracted_code_v1"][:400])


## 13. Length distribution

In [ ]:
import matplotlib.pyplot as plt

before_lengths = full_df["normalized_code"].fillna("").str.len()
after_lengths = output_df["abstracted_code_v1"].fillna("").str.len()

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(before_lengths, bins=100, alpha=0.5, label="normalized_code (before)")
ax.hist(after_lengths, bins=100, alpha=0.5, label="abstracted_code_v1 v2 (after SCoPE2)")
ax.set_xlim(0, before_lengths.quantile(0.99))
ax.set_xlabel("character length")
ax.set_ylabel("row count")
ax.set_title("Code length before/after SCoPE2 abstraction")
ax.legend()

fig_path = PREPROCESSING_OUTPUT_DIR / "scope2_length_distribution.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()

print("Saved figure to", fig_path)
print("Mean length before:", before_lengths.mean(), "| after:", after_lengths.mean())


## 14. Save report

In [ ]:
import json
from datetime import datetime, timezone

report_payload = {
    "version": s2p.SCOPE2_ABSTRACTION_VERSION,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "input_parquet": str(INPUT_PARQUET),
    "output_parquet": str(OUTPUT_PARQUET),
    "n_rows": full_report.n_rows,
    "n_empty_input": full_report.n_empty_input,
    "n_failed": full_report.n_failed,
    "failure_reason_counts": dict(full_report.failure_reason_counts),
    "skip_reason_counts": dict(full_report.skip_reason_counts),
    "elapsed_seconds": full_report.elapsed_seconds,
}

report_path = PREPROCESSING_OUTPUT_DIR / "scope2_preprocessing_report.json"
report_path.write_text(json.dumps(report_payload, indent=2), encoding="utf-8")
print("Saved report to", report_path)
print(json.dumps(report_payload, indent=2))


## 15. Downstream impact -- Track A results (EXP-0/1/2: 0.146/0.143/0.137) are now stale; see HANDOFF_scope2_preprocessing.md

## 16. Cleanup

In [ ]:
import gc
import shutil

gc.collect()

total, used, free = shutil.disk_usage(WORKSPACE_ROOT)
print(f"Disk usage at {WORKSPACE_ROOT}: {used/1e9:.1f} GB used / {total/1e9:.1f} GB total ({free/1e9:.1f} GB free)")
if used / 1e9 > STORAGE_CAP_GB:
    print(f"WARNING: workspace usage exceeds the {STORAGE_CAP_GB} GB storage cap.")
